# Robot Data Flywheel: real-data baseline

Open this notebook in Colab and select a GPU runtime. Setup downloads the project code directly from GitHub—no ZIP uploads. Dataset checks, resumable training, and development rollout evaluation follow below. This is an implementation preview; real-data performance has not yet been validated.


In [ ]:
from pathlib import Path
import subprocess, sys
REPOSITORY = "https://github.com/0-5-blood-prince/robotics-data-flywheel.git"
REVISION = "main"  # Set to a commit SHA to reproduce a particular code version.
ROOT = Path('/content/robotics-data-flywheel')
if any(name.startswith('robot_flywheel') for name in sys.modules):
    raise RuntimeError('Restart the runtime before refreshing project code. Saved Drive checkpoints remain available.')
if not ROOT.exists():
    subprocess.run(['git', 'clone', REPOSITORY, str(ROOT)], check=True)
else:
    if not (ROOT / '.git').exists():
        raise RuntimeError('This directory came from an old ZIP. Start a fresh Colab runtime once to switch to GitHub setup.')
    changes = subprocess.check_output(['git','-C',str(ROOT),'status','--porcelain','--untracked-files=no'],text=True)
    if changes.strip(): raise RuntimeError('Local code changes found; preserve them before refreshing.')
subprocess.run(['git','-C',str(ROOT),'fetch','origin',REVISION],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
CODE_COMMIT = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
print('Code commit:', CODE_COMMIT)
import os, importlib
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{ROOT}[simulation]'], check=True)
# Activate src in this already-running kernel: editable-install .pth files
# are normally processed only when a Python interpreter starts.
sys.path.insert(0, str(ROOT / 'src'))
importlib.invalidate_caches()
import robot_flywheel
assert Path(robot_flywheel.__file__).resolve().is_relative_to(ROOT.resolve())
print('Package ready:', robot_flywheel.__file__)

# Check the native simulator API before dataset preparation.
subprocess.run([sys.executable, '-c',
    "import gymnasium as gym, gym_pusht; e=gym.make('gym_pusht/PushT-v0'); e.reset(seed=0); e.step(e.action_space.sample()); e.close(); print('Simulator ready')"], check=True)


## Persistence and runtime
Set `USE_DRIVE=True` to preserve checkpoints across runtime replacement. Checkpoints are saved every 1,000 updates; rerunning training resumes automatically. Only load checkpoints produced by this project or another trusted source.

In [ ]:
import os, sys, json, torch, numpy as np
os.environ['SDL_VIDEODRIVER'] = 'dummy'
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/robotics-data-flywheel-runs')
else:
    WORK = ROOT / 'runs'
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'code_commit.txt').write_text(CODE_COMMIT)
print({'python':sys.version,'torch':torch.__version__,'cuda':torch.cuda.is_available(),
       'device':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})
if not torch.cuda.is_available():
    print('Switch to a GPU runtime before substantial training. CPU is suitable for tests only.')
!python -m unittest discover -s tests -v


## 1. Download and inspect the full-state expert dataset
The SHA-256 is recorded for provenance. Once verified, reuse it as `expected_sha256` on future downloads. It is not an independently authenticated upstream checksum.

In [ ]:
from robot_flywheel.prepare import prepare
from robot_flywheel.data import load_zarr
EPISODES, SOURCE = prepare(ROOT / 'data/pusht')
display(SOURCE)


In [ ]:
import matplotlib.pyplot as plt
example = EPISODES[0]
fig, ax = plt.subplots(1,2,figsize=(11,4))
ax[0].plot(example.states[:,0],example.states[:,1],label='Agent')
ax[0].plot(example.states[:,2],example.states[:,3],label='Object')
ax[0].set(xlim=(0,512),ylim=(0,512),title='Recorded demonstration',aspect='equal'); ax[0].legend()
ax[1].plot(example.actions); ax[1].set(title='Expert target positions',xlabel='Control step',ylabel='Position')
plt.show()


## 2. Compatibility gate
Replay three expert episodes. The legacy reset may shift object pose; the adapter reports both the raw and corrected reset. Missing velocities can still cause divergence. If this check fails, inspect the traces and fix compatibility before training; do not simply increase tolerances to get past the check.

In [ ]:
from robot_flywheel.rollout import check_replay
COMPATIBILITY = check_replay(EPISODES, WORK / 'compatibility', count=3)
display(COMPATIBILITY)
fig, axs = plt.subplots(1,3,figsize=(15,4))
for ax, episode in zip(axs, EPISODES[:3]):
    d = np.load(WORK / 'compatibility' / f'replay_{episode.id}.npz')
    ax.plot(d['recorded'][:,2],d['recorded'][:,3],label='Recorded object')
    ax.plot(d['simulated'][:,2],d['simulated'][:,3],label='Replayed object')
    ax.set(title=f'Episode {episode.id}',aspect='equal'); ax.legend()
plt.show()
assert COMPATIBILITY['passed'], 'Compatibility failed. Inspect report/traces before training.'


## 3. Train the baseline
Pilot: 50 expert episodes, 20,000 updates, batch size 128, two state observations, 16 predicted actions. EMA weights are used for evaluation. These budgets are starting hypotheses, not a promise of sufficient convergence.

The other demonstrations stay reserved for future acquisition. Normalization is fitted only on the initial training subset. Change configuration in a new run directory; extending only `updates` can resume the same run.

In [ ]:
from robot_flywheel.baseline import BaselineConfig, train_baseline, load_policy
CONFIG = BaselineConfig(initial_episodes=50, updates=20000)
RUN = WORK / 'baseline_seed42'
CHECKPOINT = train_baseline(EPISODES, SOURCE, CONFIG, RUN, compatibility=COMPATIBILITY)


In [ ]:
training = json.loads((RUN / 'training.json').read_text())
losses = np.array(training['losses'])
width = min(100,len(losses))
plt.figure(figsize=(10,3)); plt.plot(np.convolve(losses,np.ones(width)/width,mode='valid'))
plt.title('Denoising loss — moving average'); plt.xlabel('Optimizer update'); plt.show()
print('Checkpoint:', CHECKPOINT)


## 4. Development evaluation
Twenty fixed development starts, eight actions executed before replanning, 300-step limit. These seeds are **not the final test suite**. Videos show the first three seeds in order, not selected successes. Inference uses the complete DDPM schedule and may take time; results are cached per checkpoint/configuration.

In [ ]:
from robot_flywheel.rollout import evaluate
policy, obs_norm, action_norm, checkpoint_hash = load_policy(CHECKPOINT)
EVAL = RUN / f'development_{checkpoint_hash[:12]}'
summary = evaluate(policy, obs_norm, action_norm, list(range(1000,1020)), EVAL,
                   checkpoint_id=checkpoint_hash, execute=8, video_count=3)
display({k:v for k,v in summary.items() if k != 'rows'})


In [ ]:
from IPython.display import Video, display
fig, ax = plt.subplots(figsize=(9,3))
ax.bar([str(x['seed']) for x in summary['rows']], [x['max_coverage'] for x in summary['rows']])
ax.axhline(.95,color='red',linestyle='--'); ax.set(ylim=(0,1),ylabel='Maximum raw coverage',xlabel='Development seed')
plt.xticks(rotation=90); plt.show()
for path in sorted(EVAL.glob('*.mp4')):
    display(Video(str(path),embed=True,width=400))


## Interpretation and next experiment
A falling denoising loss alone does not establish control competence. Inspect rollout success, coverage, and trajectories. If performance is weak, diagnose state/action alignment, reset compatibility, action saturation, and convergence on development seeds.

This notebook stops at baseline evaluation. It does not yet run the three-round matched acquisition study, five training seeds, or held-out final evaluation. Preserve `source.json`, `dataset.json`, `config.json`, checkpoints, compatibility traces, and rollout outputs for the next stage.